# Pandas Solutions Notebook — Northwind Exercises

Το notebook περιέχει οργανωμένες ενδεικτικές λύσεις για τις αναθεωρημένες ασκήσεις Pandas πάνω στο Northwind dataset.

Υποθέσεις:
- Τα CSV αρχεία βρίσκονται σε έναν φάκελο `data` δίπλα στο notebook.
- Το notebook προσπαθεί να εντοπίσει κοινές ονομασίες αρχείων όπως `Customers.csv`, `customers.csv`, `OrderDetails.csv`, `order_details.csv` κτλ.
- Αν το dataset σου χρησιμοποιεί `Price` αντί για `UnitPrice` στον πίνακα `OrderDetails`, γίνεται αυτόματα κανονικοποίηση.

Στόχος του notebook δεν είναι μόνο να δώσει το τελικό αποτέλεσμα, αλλά και να δείξει ένα καθαρό pandas workflow:
- φόρτωση και έλεγχος δεδομένων
- φίλτρα και μετασχηματισμοί
- joins / merges
- groupby / pivot tables
- time-series και window-like λογική
- συνδυαστικές ασκήσεις με αρκετά βήματα προεργασίας

Σημείωση: επειδή αρκετές λύσεις βασίζονται σε στήλες που δημιουργούνται σε προηγούμενες ασκήσεις, καλό είναι το notebook να εκτελείται σειριακά από την αρχή προς το τέλος.

## Κεφάλαιο 1 - Φόρτωση και πρώτη επιθεώρηση

### 1. Φορτώστε τα CSV customers, orders, order_details, products, suppliers, categories, employees και shippers σε ξεχωριστά DataFrames.

In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("./data")

tables = {
    csv_file.stem: pd.read_csv(csv_file, na_values=["NULL", "NaN", ""])
    for csv_file in DATA_DIR.glob("*.csv")
}

### 2. Δημιουργήστε ένα λεξικό datasets και κατασκευάστε έναν συγκεντρωτικό πίνακα με όνομα πίνακα, αριθμό γραμμών και αριθμό στηλών για κάθε DataFrame.

In [7]:
summary_df = pd.DataFrame(
    [
        {"table": name, "rows": df.shape[0], "cols": df.shape[1]}
        for name, df in tables.items()
    ]
).sort_values("rows", ascending=False).reset_index(drop=True)

summary_df

,table,rows,cols
0,orderdetails,2155,5
1,orders,830,14
2,customers,91,14
3,products,77,10
4,suppliers,29,12
5,employees,9,18
6,categories,8,4
7,shippers,3,3


### 3. Μετατρέψτε τις στήλες OrderDate, RequiredDate, ShippedDate, BirthDate και HireDate σε datetime όπου υπάρχουν και επαληθεύστε ότι άλλαξαν τύπο.

In [16]:
tables["orders"]["OrderDate"] = pd.to_datetime(tables["orders"]["OrderDate"])
tables["orders"]["RequiredDate"] = pd.to_datetime(tables["orders"]["RequiredDate"])
tables["orders"]["ShippedDate"] = pd.to_datetime(tables["orders"]["ShippedDate"])
tables["employees"]["BirthDate"] = pd.to_datetime(tables["employees"]["BirthDate"])
tables["employees"]["HireDate"] = pd.to_datetime(tables["employees"]["HireDate"])

In [17]:

date_type_check = {
    "orders": tables["orders"][["OrderDate", "RequiredDate", "ShippedDate"]].dtypes,
    "employees": tables["employees"][["BirthDate", "HireDate"]].dtypes,
}

date_type_check

{'orders': OrderDate       datetime64[us]
 RequiredDate    datetime64[us]
 ShippedDate     datetime64[us]
 dtype: object,
 'employees': BirthDate    datetime64[us]
 HireDate     datetime64[us]
 dtype: object}

### 4. Βρείτε για κάθε DataFrame ποιες στήλες έχουν missing values και πόσα missing values έχει καθεμία.

In [21]:
missing_values_summary = {
    name: df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)
    for name, df in tables.items()
}

In [23]:
for table_name, missing_cols in missing_values_summary.items():
    if len(missing_cols) > 0:
        print(f"\n{table_name.upper()}")
        print("=" * 50)
        print(missing_cols.to_string())
    else:
        print(f"\n{table_name.upper()}: No missing values")


CUSTOMERS
Image             91
ImageThumbnail    91
age               91
Region            60
Fax               22
PostalCode         1

CATEGORIES: No missing values

PRODUCTS: No missing values

ORDERS
ShipRegion        507
ShippedDate        21
ShipPostalCode     19

SUPPLIERS
HomePage    24
Region      20
Fax         16

SHIPPERS: No missing values

ORDERDETAILS: No missing values

EMPLOYEES
Region       4
ReportsTo    1


In [26]:
customers = tables["customers"]
orders = tables["orders"]
order_details = tables["orderdetails"]
products = tables["products"]
suppliers = tables["suppliers"]
categories = tables["categories"]
employees = tables["employees"]
shippers = tables["shippers"]

### 5. Για τον πίνακα customers υπολογίστε πόσες διαφορετικές χώρες και πόσες διαφορετικές πόλεις υπάρχουν.

In [27]:
customers[["Country", "City"]].agg({"Country": "nunique", "City": "nunique"})

Country    21
City       69
dtype: int64

### 6. Εντοπίστε ποιοι πίνακες έχουν τα περισσότερα rows και εμφανίστε τους ταξινομημένους φθίνουσα ως προς το πλήθος γραμμών.

In [28]:
summary_df.sort_values("rows", ascending=False)

,table,rows,cols
0,orderdetails,2155,5
1,orders,830,14
2,customers,91,14
3,products,77,10
4,suppliers,29,12
5,employees,9,18
6,categories,8,4
7,shippers,3,3


## Κεφάλαιο 2 - Επιλογή, φίλτρα και ταξινόμηση

### 7. Εμφανίστε τις παραγγελίες που στάλθηκαν στη France ή στο Belgium.

In [ ]:
orders.loc[orders["ShipCountry"].isin(["France", "Belgium"])]

### 8. Εμφανίστε τα προϊόντα των οποίων το όνομα περιέχει τη λέξη 'queso', χωρίς διάκριση πεζών και κεφαλαίων.

In [ ]:
products.loc[products["ProductName"].str.contains("queso", case=False, na=False)]

### 9. Εμφανίστε τους υπαλλήλους με τίτλο Sales Representative που βρίσκονται στις ΗΠΑ.

In [ ]:
employees.loc[
    employees["Title"].eq("Sales Representative")
    & employees["Country"].eq("USA"),
    ["EmployeeID", "FirstName", "LastName", "Title", "Country"],
]

### 10. Εμφανίστε τα προϊόντα που χρειάζονται αναπαραγγελία, δηλαδή UnitsInStock < ReorderLevel και Discontinued = 0.

In [ ]:
products.loc[
    (products["UnitsInStock"] < products["ReorderLevel"])
    & products["Discontinued"].eq(0),
    ["ProductID", "ProductName", "UnitsInStock", "ReorderLevel", "Discontinued"],
].sort_values(["ReorderLevel", "UnitsInStock"], ascending=[False, True])

### 11. Εμφανίστε τους πελάτες που έχουν συμπληρωμένο Region και ταξινομήστε τους κατά Region και CompanyName.

In [ ]:
customers.loc[
    customers["Region"].notna(),
    ["CustomerID", "CompanyName", "City", "Region", "Country"],
].sort_values(["Region", "CompanyName"])

### 12. Εμφανίστε τα 10 ακριβότερα προϊόντα μαζί με ProductName, Price και UnitsInStock.

In [ ]:
products.loc[:, ["ProductID", "ProductName", "Price", "UnitsInStock"]].nlargest(10, "Price")

### 13. Εμφανίστε τις παραγγελίες που έγιναν μέσα στο 1997 και έχουν Freight μεγαλύτερο από το median του Freight.

In [ ]:
freight_median = orders["Freight"].median()

orders.loc[
    orders["OrderDate"].dt.year.eq(1997) & orders["Freight"].gt(freight_median),
    ["OrderID", "OrderDate", "Freight", "ShipCountry"],
].sort_values(["Freight", "OrderDate"], ascending=[False, True])

## Κεφάλαιο 3 - Καθαρισμός, μετασχηματισμοί και νέες στήλες

### 14. Δημιουργήστε στον πίνακα employees μια στήλη full_name ενώνοντας τα FirstName και LastName.

In [ ]:
employees = employees.copy()
employees["full_name"] = (
    employees["FirstName"].fillna("").str.strip()
    + " "
    + employees["LastName"].fillna("").str.strip()
).str.strip()

employees[["EmployeeID", "FirstName", "LastName", "full_name"]].head()

### 15. Δημιουργήστε στον πίνακα order_details μια στήλη line_revenue = UnitPrice * Quantity * (1 - Discount).

In [ ]:
order_details = order_details.copy()
order_details["line_revenue"] = (
    order_details["UnitPrice"] * order_details["Quantity"] * (1 - order_details["Discount"])
)

order_details.head()

### 16. Δημιουργήστε στον πίνακα products μια στήλη stock_gap = ReorderLevel - UnitsInStock.

In [ ]:
products = products.copy()
products["stock_gap"] = products["ReorderLevel"] - products["UnitsInStock"]

products[["ProductID", "ProductName", "UnitsInStock", "ReorderLevel", "stock_gap"]].head()

### 17. Δημιουργήστε μια κατηγορική στήλη price_category με τιμές Cheap, Medium και Expensive ανάλογα με την τιμή του προϊόντος.

In [ ]:
products["price_category"] = pd.cut(
    products["Price"],
    bins=[-np.inf, 20, 50, np.inf],
    labels=["Cheap", "Medium", "Expensive"],
)

products[["ProductID", "ProductName", "Price", "price_category"]].head(10)

### 18. Δημιουργήστε στον πίνακα orders μια στήλη shipping_delay_days = ShippedDate - OrderDate σε ημέρες.

In [ ]:
orders = orders.copy()
orders["shipping_delay_days"] = (orders["ShippedDate"] - orders["OrderDate"]).dt.days

orders[["OrderID", "OrderDate", "ShippedDate", "shipping_delay_days"]].head()

### 19. Δημιουργήστε στον πίνακα orders μια boolean στήλη is_late που να δείχνει αν μια παραγγελία στάλθηκε μετά το RequiredDate.

In [ ]:
orders["is_late"] = orders["ShippedDate"] > orders["RequiredDate"]

orders[["OrderID", "RequiredDate", "ShippedDate", "is_late"]].head()

### 20. Δημιουργήστε στον πίνακα customers μια στήλη has_fax που να παίρνει τιμή 1 όταν υπάρχει fax και 0 όταν δεν υπάρχει.

In [ ]:
customers = customers.copy()
customers["has_fax"] = customers["Fax"].notna().astype(int)

customers[["CustomerID", "CompanyName", "Fax", "has_fax"]].head()

### 21. Δημιουργήστε ένα αντίγραφο του customers στο οποίο τα missing values της Region αντικαθίστανται με 'Unknown' και εμφανίστε πόσες εγγραφές άλλαξαν.

In [ ]:
customers_filled = customers.copy()
changed_rows = customers_filled["Region"].isna().sum()
customers_filled["Region"] = customers_filled["Region"].fillna("Unknown")

changed_rows

## Κεφάλαιο 4 - GroupBy και συγκεντρωτικά

### 22. Υπολογίστε πόσοι πελάτες υπάρχουν συνολικά.

In [ ]:
customers.shape[0]

### 23. Υπολογίστε πόσοι πελάτες υπάρχουν ανά χώρα και ταξινομήστε φθίνουσα.

In [ ]:
customers.groupby("Country").size().sort_values(ascending=False).rename("customer_count")

### 24. Υπολογίστε πόσοι πελάτες υπάρχουν ανά ContactTitle.

In [ ]:
customers.groupby("ContactTitle").size().sort_values(ascending=False).rename("customer_count")

### 25. Υπολογίστε για κάθε κατηγορία προϊόντων το πλήθος προϊόντων και τη μέση τιμή τους.

In [ ]:
products_with_categories = products.merge(
    categories[["CategoryID", "CategoryName"]],
    on="CategoryID",
    how="left",
)

products_with_categories.groupby("CategoryName", as_index=False).agg(
    products_count=("ProductID", "count"),
    avg_price=("Price", "mean"),
).sort_values("avg_price", ascending=False)

### 26. Υπολογίστε το συνολικό revenue ανά προϊόν με βάση το order_details και ταξινομήστε από το μεγαλύτερο στο μικρότερο.

In [ ]:
revenue_per_product = (
    order_details
    .merge(products[["ProductID", "ProductName"]], on="ProductID", how="left")
    .groupby(["ProductID", "ProductName"], as_index=False)["line_revenue"]
    .sum()
    .sort_values("line_revenue", ascending=False)
)

revenue_per_product

### 27. Υπολογίστε για κάθε supplier πόσα διαφορετικά προϊόντα παρέχει και ποια είναι η μέση τιμή των προϊόντων του.

In [ ]:
supplier_product_stats = (
    products
    .groupby("SupplierID", as_index=False)
    .agg(
        distinct_products=("ProductID", "nunique"),
        avg_product_price=("Price", "mean"),
    )
    .merge(suppliers[["SupplierID", "CompanyName"]], on="SupplierID", how="left")
    .sort_values(["distinct_products", "avg_product_price"], ascending=[False, False])
)

supplier_product_stats

### 28. Υπολογίστε για κάθε χώρα αποστολής πόσες παραγγελίες υπάρχουν και ποιο είναι το μέσο Freight.

In [ ]:
orders.groupby("ShipCountry", as_index=False).agg(
    orders_count=("OrderID", "count"),
    avg_freight=("Freight", "mean"),
).sort_values("orders_count", ascending=False)

### 29. Υπολογίστε για κάθε υπάλληλο τον αριθμό παραγγελιών που χειρίστηκε και το συνολικό revenue που του αντιστοιχεί.

In [ ]:
employee_order_revenue = (
    orders[["OrderID", "EmployeeID"]]
    .merge(order_details[["OrderID", "line_revenue"]], on="OrderID", how="left")
    .groupby("EmployeeID", as_index=False)
    .agg(
        orders_count=("OrderID", "nunique"),
        total_revenue=("line_revenue", "sum"),
    )
    .merge(employees[["EmployeeID", "full_name"]], on="EmployeeID", how="left")
    .sort_values("total_revenue", ascending=False)
)

employee_order_revenue

## Κεφάλαιο 5 - Merges και συνδυασμός πινάκων

### 30. Συνδέστε τα products με τα suppliers και εμφανίστε το ProductName και το όνομα του supplier.

In [ ]:
products.merge(
    suppliers[["SupplierID", "CompanyName"]],
    on="SupplierID",
    how="left",
)[["ProductID", "ProductName", "CompanyName"]].rename(columns={"CompanyName": "SupplierName"})

### 31. Συνδέστε τα orders με τα customers και εμφανίστε OrderID, CompanyName, OrderDate και ShipCountry.

In [ ]:
orders.merge(
    customers[["CustomerID", "CompanyName"]],
    on="CustomerID",
    how="left",
)[["OrderID", "CompanyName", "OrderDate", "ShipCountry"]].sort_values("OrderDate")

### 32. Συνδέστε τα order_details με τα products και εμφανίστε για κάθε γραμμή παραγγελίας το ProductName, Quantity, UnitPrice και line_revenue.

In [ ]:
order_details.merge(
    products[["ProductID", "ProductName"]],
    on="ProductID",
    how="left",
)[["OrderID", "ProductName", "Quantity", "UnitPrice", "line_revenue"]]

### 33. Δημιουργήστε έναν ενιαίο πίνακα order_lines που να περιέχει OrderID, OrderDate, Customer CompanyName, Employee full_name, ProductName, CategoryName, Quantity, Discount και line_revenue.

In [ ]:
order_lines = (
    orders[["OrderID", "CustomerID", "EmployeeID", "OrderDate", "ShipCountry", "shipping_delay_days", "is_late"]]
    .merge(customers[["CustomerID", "CompanyName", "Country"]], on="CustomerID", how="left")
    .merge(employees[["EmployeeID", "full_name"]], on="EmployeeID", how="left")
    .merge(order_details[["OrderID", "ProductID", "Quantity", "Discount", "UnitPrice", "line_revenue"]], on="OrderID", how="left")
    .merge(products[["ProductID", "ProductName", "CategoryID", "SupplierID"]], on="ProductID", how="left")
    .merge(categories[["CategoryID", "CategoryName"]], on="CategoryID", how="left")
)

order_lines[[
    "OrderID", "OrderDate", "CompanyName", "full_name",
    "ProductName", "CategoryName", "Quantity", "Discount", "line_revenue"
]].head(20)

### 34. Εντοπίστε τους customers που δεν έχουν κάνει καμία παραγγελία.

In [ ]:
customers.merge(
    orders[["CustomerID"]].drop_duplicates(),
    on="CustomerID",
    how="left",
    indicator=True,
).query("_merge == 'left_only'")[["CustomerID", "CompanyName", "Country"]]

### 35. Εντοπίστε τα προϊόντα που δεν έχουν εμφανιστεί ποτέ σε καμία παραγγελία.

In [ ]:
products.merge(
    order_details[["ProductID"]].drop_duplicates(),
    on="ProductID",
    how="left",
    indicator=True,
).query("_merge == 'left_only'")[["ProductID", "ProductName", "Price"]]

### 36. Για κάθε παραγγελία υπολογίστε το total_order_value και ενώστε το αποτέλεσμα με το όνομα του πελάτη και του υπαλλήλου.

In [ ]:
order_totals = (
    order_lines
    .groupby(["OrderID", "CompanyName", "full_name"], as_index=False)["line_revenue"]
    .sum()
    .rename(columns={"CompanyName": "CustomerName", "full_name": "EmployeeName", "line_revenue": "total_order_value"})
    .sort_values("total_order_value", ascending=False)
)

order_totals.head(20)

### 37. Εμφανίστε τους suppliers των οποίων προϊόντα ανήκουν στην κατηγορία Beverages. Να εμφανιστεί κάθε supplier μόνο μία φορά.

In [ ]:
products.merge(categories[["CategoryID", "CategoryName"]], on="CategoryID", how="left") \
    .merge(suppliers[["SupplierID", "CompanyName"]], on="SupplierID", how="left") \
    .loc[lambda df: df["CategoryName"].eq("Beverages"), ["SupplierID", "CompanyName"]] \
    .drop_duplicates() \
    .sort_values("CompanyName")

## Κεφάλαιο 6 - Pivot tables και reshaping

### 38. Δημιουργήστε pivot table που να δείχνει το συνολικό revenue ανά ShipCountry.

In [ ]:
pd.pivot_table(
    order_lines,
    values="line_revenue",
    index="ShipCountry",
    aggfunc="sum",
).sort_values("line_revenue", ascending=False)

### 39. Δημιουργήστε pivot table που να δείχνει revenue ανά ShipCountry και CategoryName.

In [ ]:
pd.pivot_table(
    order_lines,
    values="line_revenue",
    index="ShipCountry",
    columns="CategoryName",
    aggfunc="sum",
    fill_value=0,
)

### 40. Δημιουργήστε pivot table που να δείχνει πόσες παραγγελίες χειρίστηκε κάθε employee ανά έτος.

In [ ]:
orders_by_year = orders.assign(OrderYear=orders["OrderDate"].dt.year)

pd.pivot_table(
    orders_by_year,
    values="OrderID",
    index="EmployeeID",
    columns="OrderYear",
    aggfunc="count",
    fill_value=0,
)

### 41. Εμφανίστε τις 5 χώρες με το μεγαλύτερο revenue χρησιμοποιώντας pivot_table().

In [ ]:
pd.pivot_table(
    order_lines,
    values="line_revenue",
    index="ShipCountry",
    aggfunc="sum",
).sort_values("line_revenue", ascending=False).head(5)

### 42. Δημιουργήστε έναν πίνακα με τα έτη στις γραμμές, τους μήνες στις στήλες και το πλήθος παραγγελιών στις τιμές.

In [ ]:
orders_year_month = orders.assign(
    OrderYear=orders["OrderDate"].dt.year,
    OrderMonth=orders["OrderDate"].dt.month,
)

pd.pivot_table(
    orders_year_month,
    values="OrderID",
    index="OrderYear",
    columns="OrderMonth",
    aggfunc="count",
    fill_value=0,
)

### 43. Δημιουργήστε πίνακα συχνοτήτων που να δείχνει πόσα προϊόντα έχει κάθε supplier σε κάθε κατηγορία.

In [ ]:
supplier_category_counts = products.merge(
    categories[["CategoryID", "CategoryName"]],
    on="CategoryID",
    how="left",
).merge(
    suppliers[["SupplierID", "CompanyName"]],
    on="SupplierID",
    how="left",
)

pd.crosstab(
    supplier_category_counts["CompanyName"],
    supplier_category_counts["CategoryName"],
)

## Κεφάλαιο 7 - Χρόνος, window-like λογική και ακολουθίες

### 44. Υπολογίστε πόσες παραγγελίες έγιναν ανά μήνα.

In [ ]:
orders_per_month = (
    orders
    .set_index("OrderDate")
    .resample("M")["OrderID"]
    .count()
    .rename("orders_count")
)

orders_per_month

### 45. Βρείτε ποιος μήνας είχε το μεγαλύτερο συνολικό revenue.

In [ ]:
monthly_revenue = (
    orders[["OrderID", "OrderDate"]]
    .merge(order_details[["OrderID", "line_revenue"]], on="OrderID", how="left")
    .set_index("OrderDate")
    .resample("M")["line_revenue"]
    .sum()
    .rename("monthly_revenue")
)

monthly_revenue.sort_values(ascending=False).head(1)

### 46. Υπολογίστε τη μέση καθυστέρηση αποστολής σε ημέρες και εμφανίστε τις 10 μεγαλύτερες καθυστερήσεις.

In [ ]:
average_delay = orders["shipping_delay_days"].mean()

longest_delays = orders.loc[
    orders["shipping_delay_days"].notna(),
    ["OrderID", "OrderDate", "ShippedDate", "shipping_delay_days"],
].sort_values("shipping_delay_days", ascending=False).head(10)

average_delay, longest_delays

### 47. Αριθμήστε τις παραγγελίες κάθε πελάτη χρονικά, δημιουργώντας στήλη order_number_per_customer.

In [ ]:
orders_sorted = orders.sort_values(["CustomerID", "OrderDate", "OrderID"]).copy()
orders_sorted["order_number_per_customer"] = orders_sorted.groupby("CustomerID").cumcount() + 1

orders_sorted[["CustomerID", "OrderID", "OrderDate", "order_number_per_customer"]].head(20)

### 48. Υπολογίστε το cumulative revenue ανά μήνα και μια moving average 3 μηνών.

In [ ]:
monthly_revenue_df = monthly_revenue.to_frame()
monthly_revenue_df["cumulative_revenue"] = monthly_revenue_df["monthly_revenue"].cumsum()
monthly_revenue_df["rolling_3m_avg"] = monthly_revenue_df["monthly_revenue"].rolling(3).mean()

monthly_revenue_df

### 49. Για κάθε πελάτη υπολογίστε τις ημέρες που πέρασαν από την προηγούμενη παραγγελία του.

In [ ]:
customer_orders = orders.sort_values(["CustomerID", "OrderDate", "OrderID"]).copy()
customer_orders["days_since_previous_order"] = (
    customer_orders.groupby("CustomerID")["OrderDate"].diff().dt.days
)

customer_orders[["CustomerID", "OrderID", "OrderDate", "days_since_previous_order"]].head(20)

## Κεφάλαιο 8 - Συνδυαστικές ασκήσεις αυξανόμενης δυσκολίας

### 50. Εμφανίστε τους 10 πελάτες με τη μεγαλύτερη συνολική δαπάνη μαζί με πλήθος παραγγελιών και μέση αξία παραγγελίας.

In [ ]:
customer_order_totals = (
    order_lines
    .groupby(["CustomerID", "CompanyName", "Country", "OrderID"], as_index=False)["line_revenue"]
    .sum()
    .rename(columns={"line_revenue": "order_value"})
)

top_10_customers = (
    customer_order_totals
    .groupby(["CustomerID", "CompanyName", "Country"], as_index=False)
    .agg(
        total_spend=("order_value", "sum"),
        orders_count=("OrderID", "nunique"),
        avg_order_value=("order_value", "mean"),
    )
    .sort_values("total_spend", ascending=False)
    .head(10)
)

top_10_customers

### 51. Υπολογίστε για κάθε υπάλληλο το ποσοστό καθυστερημένων παραγγελιών και εμφανίστε τους 5 πρώτους.

In [ ]:
late_rate_per_employee = (
    orders
    .groupby("EmployeeID", as_index=False)
    .agg(
        total_orders=("OrderID", "count"),
        late_orders=("is_late", "sum"),
    )
    .merge(employees[["EmployeeID", "full_name"]], on="EmployeeID", how="left")
)

late_rate_per_employee["late_order_rate"] = (
    late_rate_per_employee["late_orders"] / late_rate_per_employee["total_orders"]
)

late_rate_per_employee.sort_values("late_order_rate", ascending=False).head(5)

### 52. Βρείτε ποια κατηγορία έχει το μεγαλύτερο συνολικό revenue και μέσα σε αυτήν εμφανίστε τα 3 προϊόντα με το μεγαλύτερο revenue.

In [ ]:
category_revenue = (
    order_lines
    .groupby("CategoryName", as_index=False)["line_revenue"]
    .sum()
    .sort_values("line_revenue", ascending=False)
)

top_category = category_revenue.iloc[0]["CategoryName"]

top_3_products_in_top_category = (
    order_lines.loc[order_lines["CategoryName"].eq(top_category)]
    .groupby(["CategoryName", "ProductName"], as_index=False)["line_revenue"]
    .sum()
    .sort_values("line_revenue", ascending=False)
    .head(3)
)

top_category, top_3_products_in_top_category

### 53. Εμφανίστε τους πελάτες που έχουν αγοράσει προϊόντα από περισσότερες από 3 διαφορετικές κατηγορίες και έχουν συνολική δαπάνη μεγαλύτερη από 5000.

In [ ]:
customer_category_spend = (
    order_lines
    .groupby(["CustomerID", "CompanyName"], as_index=False)
    .agg(
        distinct_categories=("CategoryName", "nunique"),
        total_spend=("line_revenue", "sum"),
    )
)

customer_category_spend.loc[
    (customer_category_spend["distinct_categories"] > 3)
    & (customer_category_spend["total_spend"] > 5000)
].sort_values("total_spend", ascending=False)

### 54. Δημιουργήστε πίνακα απόδοσης προμηθευτών με τις στήλες SupplierName, DistinctProductsSold, TotalQuantitySold και TotalRevenue.

In [ ]:
supplier_sales_agg = (
    order_lines
    .groupby("SupplierID", as_index=False)
    .agg(
        DistinctProductsSold=("ProductID", "nunique"),
        TotalQuantitySold=("Quantity", "sum"),
        TotalRevenue=("line_revenue", "sum"),
    )
)

supplier_performance = (
    suppliers[["SupplierID", "CompanyName"]]
    .merge(supplier_sales_agg, on="SupplierID", how="left")
    .rename(columns={"CompanyName": "SupplierName"})
)

supplier_performance[["DistinctProductsSold", "TotalQuantitySold", "TotalRevenue"]] = (
    supplier_performance[["DistinctProductsSold", "TotalQuantitySold", "TotalRevenue"]]
    .fillna(0)
)

supplier_performance.sort_values("TotalRevenue", ascending=False)

### 55. Βρείτε τη χώρα με το μεγαλύτερο average Freight, αλλά να ληφθούν υπόψη μόνο χώρες με τουλάχιστον 10 παραγγελίες.

In [ ]:
country_freight_stats = (
    orders.groupby("ShipCountry", as_index=False)
    .agg(
        orders_count=("OrderID", "count"),
        avg_freight=("Freight", "mean"),
    )
)

country_freight_stats.loc[
    country_freight_stats["orders_count"] >= 10
].sort_values("avg_freight", ascending=False).head(1)

### 56. Δημιουργήστε segmentation πελατών σε 3 επίπεδα, Low, Medium και High, με βάση το total spend και εμφανίστε πόσοι πελάτες ανήκουν σε κάθε segment.

In [ ]:
customer_spend = (
    customer_order_totals
    .groupby(["CustomerID", "CompanyName"], as_index=False)["order_value"]
    .sum()
    .rename(columns={"order_value": "total_spend"})
)

customer_spend["segment"] = pd.qcut(
    customer_spend["total_spend"].rank(method="first"),
    q=3,
    labels=["Low", "Medium", "High"],
)

customer_spend.groupby("segment").size().rename("customers_count")

### 57. Για κάθε μήνα και employee υπολογίστε το revenue και το ποσοστό συμμετοχής του employee στο συνολικό revenue του μήνα.

In [ ]:
employee_month_revenue = (
    order_lines
    .assign(RevenueMonth=order_lines["OrderDate"].dt.to_period("M").dt.to_timestamp())
    .groupby(["RevenueMonth", "EmployeeID", "full_name"], as_index=False)["line_revenue"]
    .sum()
    .rename(columns={"line_revenue": "employee_month_revenue"})
)

employee_month_revenue["month_total_revenue"] = (
    employee_month_revenue.groupby("RevenueMonth")["employee_month_revenue"].transform("sum")
)

employee_month_revenue["revenue_share_pct"] = (
    100 * employee_month_revenue["employee_month_revenue"] / employee_month_revenue["month_total_revenue"]
)

employee_month_revenue.sort_values(["RevenueMonth", "employee_month_revenue"], ascending=[True, False])

### 58. Δημιουργήστε ένα τελικό analytical DataFrame σε επίπεδο παραγγελίας που να περιέχει total_order_value, number_of_line_items, total_quantity, shipping_delay_days, is_late, customer country και employee full_name.

In [ ]:
order_level_analytics = (
    order_lines
    .groupby(
        ["OrderID", "OrderDate", "CustomerID", "CompanyName", "Country", "EmployeeID", "full_name", "shipping_delay_days", "is_late"],
        as_index=False,
    )
    .agg(
        total_order_value=("line_revenue", "sum"),
        number_of_line_items=("ProductID", "size"),
        total_quantity=("Quantity", "sum"),
    )
    .rename(
        columns={
            "CompanyName": "customer_name",
            "Country": "customer_country",
            "full_name": "employee_full_name",
        }
    )
    .sort_values("total_order_value", ascending=False)
)

order_level_analytics.head(20)

## Γρήγορα tips

- Για καθαρό filtering προτίμησε `df.loc[...]`.
- Για SQL-like joins χρησιμοποίησε `merge()`.
- Για aggregations: `groupby()`, `agg()`, `transform()`.
- Για reshape/pivots: `pivot_table()` και `crosstab()`.
- Για χρόνο: `pd.to_datetime()`, `dt`, `resample()`, `to_period()`.
- Για window-like λογική: `cumcount()`, `diff()`, `cumsum()`, `rolling()`, `rank()`, `shift()`.